# Send and receive a pulse (tProc v2)

This notebook is based on the tProc v2 tutorial [`01_Basic_Sequencing.ipynb`](../../../docs/source/tutorials/01_Basic_Sequencing.ipynb), adapted to load a custom bitstream built from this repo and (once available) score each pulse with the FPGA-resident NN classifier, as in the tProc v1 branch's [`qick_ml/send_receive_pulse.ipynb`](https://github.com/GiuseppeDiGuglielmo/qick/blob/ml-integration-tproc-v1-2026/qick_ml/send_receive_pulse.ipynb).

In [ ]:
# You should be on an ml branch (e.g. ml-integration-tproc-v2-2026) of https://github.com/GiuseppeDiGuglielmo/qick
!git branch

In [ ]:
!echo -n "git rev:  "; git rev-parse --short HEAD
!echo -n "git date: "; git log -1 --format=%cd

In [ ]:
import subprocess
branch_name = subprocess.getoutput("git rev-parse --abbrev-ref HEAD").replace('/', '-')
print(branch_name)

In [ ]:
# Use this repo's own qick_lib (its driver bindto strings and IP classes match
# this branch's custom firmware/bitstream) instead of whatever qick package
# happens to be installed system-wide via pip.
import sys, os
repo_qick_lib = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'qick_lib'))
if repo_qick_lib not in sys.path:
    sys.path.insert(0, repo_qick_lib)

# Import the QICK drivers and auxiliary libraries
from qick import *
# tProc v2 programs are built on AveragerProgramV2, not the tProc v1 AveragerProgram
from qick.asm_v2 import AveragerProgramV2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import qick
print('Using qick from:', qick.__file__)

In [ ]:
# Current directory
!pwd

In [ ]:
# List custom bitstreams
!ls -lhR 216/$branch_name

In [ ]:
# Choose custom bitstream
# Filenames match what `make copy` (firmware/tools) pushes from a design's out/
# directory: firmware/projects/qick_tprocv2_216_standard_1ch/out/*.bit

# QICK tProc v2
# CUSTOM_BIT = '216/{}/qick_216_tprocv2.bit'.format(branch_name)
# HAS_NN = 0

# QICK tProc v2 + readout ILAs
CUSTOM_BIT = '216/{}/qick_216_tprocv2_ila.bit'.format(branch_name)
HAS_NN = 0

# QICK tProc v2 + NN classifier (not yet built for tProc v2; placeholder for
# when the classifier IP from the tProc v1 branch is ported here)
# CUSTOM_BIT = '216/{}/qick_216_tprocv2_nn.bit'.format(branch_name)
# HAS_NN = 1

# Configure channels
GEN_CH = 0
RO_CH = 0

In [ ]:
# Load bitstream with custom overlay
if 'CUSTOM_BIT' not in locals():
    HAS_NN = 0  # the default bitstream has no NN
    soc = QickSoc()
else:
    import os
    # Normalize path
    CUSTOM_BIT_FULL_PATH = os.path.normpath(os.getcwd() + '/' + CUSTOM_BIT)
    print('Custom bitstream:', CUSTOM_BIT_FULL_PATH)

    # The HWH file must sit next to the BIT file
    hwh_path = os.path.splitext(CUSTOM_BIT_FULL_PATH)[0] + '.hwh'
    if not os.path.isfile(hwh_path):
        raise FileNotFoundError('Missing HWH file next to the bitstream: ' + hwh_path)

    soc = QickSoc(bitfile=CUSTOM_BIT_FULL_PATH)

soccfg = soc
print(soccfg)

In [ ]:
print('Loaded bitstream:', soccfg.bitfile_name)

### Hardware Configuration

<!--generator channel 0 <-> readout channel 0, per qick_tprocv2_216_standard_1ch-->

In [ ]:
print('Generator channel {}: {}'.format(GEN_CH, soccfg._describe_dac(soccfg['gens'][GEN_CH]['dac'])))
print('Readout channel   {}: {}'.format(RO_CH, soccfg._describe_adc(soccfg['readouts'][RO_CH]['adc'])))

In [ ]:
# Single constant pulse and readout, as in "A Simple Single-Pulse Program" of
# docs/source/tutorials/01_Basic_Sequencing.ipynb
class SinglePulseProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        # Declare the generator and readout channels
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)  # nqz=1 means no frequency folding
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Define a constant pulse (no envelope shaping)
        self.add_pulse(
            ch=cfg['gen_ch'],
            name="my_pulse",
            style="const",           # Constant amplitude
            freq=cfg['freq'],        # Frequency in MHz
            length=cfg['pulse_len'], # Duration in microseconds
            phase=cfg['phase'],      # Phase in degrees
            gain=cfg['gain']         # Amplitude (0 to 1)
        )

        # Configure the readout
        self.add_readoutconfig(
            ch=cfg['ro_ch'],
            name="my_ro",
            freq=cfg['freq'],
            gen_ch=cfg['gen_ch']
        )
        self.send_readoutconfig(ch=cfg['ro_ch'], name="my_ro", t=0)

    def _body(self, cfg):
        # Play the pulse at time t=0
        self.pulse(ch=cfg['gen_ch'], name="my_pulse", t=0)
        # Trigger the readout
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])

### NN scaler setup

### Load an excited state

In [ ]:
# Configure the experiment
config = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 500.0,          # MHz
    'pulse_len': 0.1,       # microseconds
    'phase': 0,             # degrees
    'gain': 1.0,            # full amplitude
    'trig_time': 0.2,       # microseconds (when to start readout)
    'ro_len': 0.5,          # microseconds (readout duration)
    'rounds': 10,           # averages, passed to acquire_decimated
}

In [ ]:
#########################################################
# If you are running Vivado ILA, it is time to arm them #
#########################################################

In [ ]:
import pynq
print('PYNQ version: {}'.format(pynq.__version__))

In [ ]:
from pynq import MMIO

if HAS_NN:
    nn_config = soccfg.NN_0

    mmio_nn_config = MMIO(
        nn_config.mmio.base_addr,
        nn_config.mmio.length
    )

    print('NN0 config @{} {:4}KB {:8}w'.format(
        hex(nn_config.mmio.base_addr),
        int(nn_config.mmio.length/1024),
        len(nn_config.mmio.array)
    ))
    print('  - window_size     @{}'.format((nn_config.register_map.window_size.address)))
    print('  - window_offset   @{}'.format((nn_config.register_map.window_offset.address)))
    print('  - scaling_factor  @{}'.format((nn_config.register_map.scaling_factor.address)))
    print('  - out_reset       @{}'.format((nn_config.register_map.out_reset.address)))
    print('  - out_offset      @{}'.format((nn_config.register_map.out_offset.address)))
    print('      - value        {}'.format(mmio_nn_config.read(
                                        nn_config.register_map.out_offset.address)))
    print('  - out_offset_ctrl @{}'.format((nn_config.register_map.out_offset_ctrl.address)))
else:
    print('No NN')

In [ ]:
import qick_ml_lib
from qick_ml_lib import (to_float, reset_classifier, configure_classifier,
                          get_classifier_prediction_count, get_classifier_prediction,
                          get_classifier_predictions, print_classifier_buffer)

# The classifier helpers live in qick_ml_lib.py and need the SoC handle registered once
qick_ml_lib.set_soccfg(soccfg)

In [ ]:
%%time
if HAS_NN:
    if 'WINDOW_SIZE' not in globals():
        WINDOW_SIZE = 400  # default, unless set in the bitstream cell
    WINDOW_OFFSET = 5
    SCALING_FACTOR = 1

    # Reset the classifier in deep mode (first 16 predictions, i.e. 32 words, of the buffer are set to 0)
    reset_classifier(deep_reset=True, index_lo=0, index_hi=15)

    # The NN IP ignores the WINDOW_SIZE for now
    configure_classifier(WINDOW_SIZE, WINDOW_OFFSET, SCALING_FACTOR, debug=True)

In [ ]:
if HAS_NN:
    print_classifier_buffer(index_lo=0, index_hi=7)

In [ ]:
%%time
# Create and run the program
prog = SinglePulseProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# Acquire data (decimated mode gives IQ traces vs time)
iq_data = prog.acquire_decimated(soc, rounds=config['rounds'])
time_axis = prog.get_time_axis(ro_index=0)

# Plot the results
plt.figure(figsize=(10, 5))
if HAS_NN:
    plt.axvline(WINDOW_OFFSET, ls='--', color='red')
    plt.axvline(WINDOW_OFFSET + WINDOW_SIZE, ls='--', color='red')
plt.plot(time_axis, iq_data[0][:,0], label='I (In-phase)')
plt.plot(time_axis, iq_data[0][:,1], label='Q (Quadrature)')
plt.plot(time_axis, np.abs(iq_data[0].dot([1,1j])), label='Magnitude')
plt.legend()
plt.xlabel('Time (µs)')
plt.ylabel('Amplitude (a.u.)')
plt.title('Single Pulse Measurement, averages = {}'.format(config['rounds']))
plt.grid(True)
plt.show()

In [ ]:
from qick_ml_lib import float_to_hex32, int_to_twos_complement_hex32

# Take the trace from the first (and only) readout channel.
iq = iq_data[0]

# Interleave I and Q into a single flat sequence: I[0], Q[0], I[1], Q[1], ...
# (iq is an Nx2 array of [I, Q] rows, so flattening it in row-major order already interleaves them)
iq_sequence = [int(v) for v in iq.flatten()]

# Encode each sample as hex: as an IEEE-754 float, and as a 32b two's-complement int
hex_iq_sequence_flt = [float_to_hex32(num) for num in iq_sequence]
hex_iq_sequence_dec = [int_to_twos_complement_hex32(int(i)) for i in iq_sequence]

print('Sequence length:', len(iq_sequence))

# Debug: uncomment to dump the raw I/Q arrays or the encoded hex sequences
# print('I (lo, data[15:0])')
# print(iq[:, 0])
# print('Q (hi, data[31:16])')
# print(iq[:, 1])
# print(iq_sequence)
# print(hex_iq_sequence_flt)
# print(hex_iq_sequence_dec)

for i in range(0, len(iq_sequence), 2):
    print("I[{}] {:4.0f} {}".format(i//2, iq_sequence[i], hex_iq_sequence_dec[i]))
    print("Q[{}] {:4.0f} {}".format(i//2, iq_sequence[i+1], hex_iq_sequence_dec[i+1]))

In [ ]:
#
# ATTENTION: if the cell fails, please make sure that
#            the base_address matches the associated
#            value in the address editor of the Vivado
#            project.
#
if HAS_NN:
    for i in range(1):
        ground_state_logit, excited_state_logit = get_classifier_prediction(i)

        if to_float(ground_state_logit) > to_float(excited_state_logit):
            print("Prediction: ground state")
        else:
            print("Prediction: excited state")
        print('Logit values as int: [{}, {}]'.format(ground_state_logit, excited_state_logit))
        print('Logit values as hex: [{}, {}]'.format(hex(ground_state_logit), hex(excited_state_logit)))
        print('Logit values as flt: [{}, {}]'.format(to_float(ground_state_logit), to_float(excited_state_logit)))
else:
    print('No NN in this bitstream!')

In [ ]:
if HAS_NN:
    print_classifier_buffer(index_lo=0, index_hi=7)

In [ ]:
if HAS_NN:
    reset_classifier(deep_reset=True, index_hi=7)

In [ ]:
if HAS_NN:
    print_classifier_buffer(index_lo=0, index_hi=7)

In [ ]:
if HAS_NN:
    print('Readout count: {}'.format(get_classifier_prediction_count()))